# 5번 에이전트: 대상 기반 감성분석 (여러 기업, 파인튜닝 없음) — 4차


## 4차의 범위와 평가 순서 (2026-10-27)

- 2·3차 모델을 기준선으로 두고, 별도 소액주주 관점 dev 112문장에서 지배구조 대상+행동 패턴과 조정 강도를 정한다.
- dev는 가상 기업 10곳으로 구성한다. 점검 세트와 원본 test의 라벨/성능은 튜닝에 쓰지 않고, 노트북 끝의 최종 평가 절에서만 확인한다.
- 정규식은 `gov_features` 정의 시점의 객체를 기본 인자로 보관해, 이후 `RX_*` 재할당이 3차 결과를 바꾸지 않게 한다.
- 규칙 패턴을 dev 전체를 보고 작성했으므로 그룹 교차검증 수치는 참고값이다. 외부 점검 세트는 최종 방향 확인용이며 문장 수가 작다.


## 0-1. 전체 구조에서 이 노트북의 위치 (팀원용)

팀 구조도에서 5번이 맡은 칸은 **"경영진·대주주: 경영 행위·지배구조"** 다. 이 노트북은 그 칸 안의 **감성 판정 부품**이다.

```text
사용자 질문
   │
① Planner ── 필요한 관점·자료·과업을 정함 ("경영진·대주주 관점 필요")
   │
② Researcher ── 공시·뉴스 문장을 모음 (DART, 뉴스 제목)
   │
   ▼
┌──────────────── 6개 분석 관점 ────────────────┐
│ 투자심리·수급 │ 거시경제 │ 가치·위험 │ 산업·사업모델 │ 기술적 분석 │
│                                              │
│  [경영진·대주주]  ← 5번                        │
│    ├ 사실 판정: 지분율·등기 여부·자사주 (규칙 13개, 별도 노트북)
│    └ 감성 판정: 문장 속 경영진·대주주에 대한 긍정/중립/부정  ← 이 노트북
└──────────────────────────────────────────────┘
   │  판정 + 근거 문장 + 출처 + 확신도
   ▼
③ Verifier ── 원문·계산·주장별 근거 확인 → 조율 관점 → 최종 Verifier → 사용자 보고서
```

### 투자심리·수급 관점과 겹치지 않는 이유
| 구분 | 투자심리·수급 | 경영진·대주주(이 노트북) |
| --- | --- | --- |
| 무엇의 감성인가 | 종목 전체의 분위기, 투자자별 거래 | 문장 속 **특정 대상**(경영진·대주주·기업)에 대한 평가 |
| 예: "실적 호조, 최대주주는 지분 매각" | 종목 분위기: 엇갈림 | 대주주에 대한 신호: 부정 |

같은 기사라도 **보는 대상이 다르다.** 그래서 두 관점이 같은 숫자를 두 번 세지 않는다.

### Verifier에 넘기는 것
판정만 넘기지 않는다. ③ Verifier가 확인할 수 있게 다음을 함께 넘긴다.
- **쓴 구절**: 원문 중 어느 부분을 봤는지
- **모델별 확률**: 판단의 근거
- **확신도**: 얼마나 확실한지. 확신이 낮으면 "자료 없음"으로 보내고, 구조도의 "해결 가능한 근거 부족" 경로로 재조사를 요청할 수 있다.

## 0-2. 모델 구성과 근거 (팀원용)

### 한 줄 요약
**성격이 다른 모델 4개에게 각자 잘하는 질문을 하고, 그 답을 위원장(보정 층)이 종합한다.**
모델 가중치는 하나도 바꾸지 않는다(파인튜닝 없음).

### 왜 모델 하나로는 부족한가
금융 감성 모델은 대부분 **문장 전체**의 감성을 낸다. 그런데 5번이 알고 싶은 것은 **특정 대상**의 감성이다.
- "A사 흑자전환, B사 적자 지속" → 문장 전체는 "엇갈림". B사에 대해서는 "부정"이어야 한다.
- "대표 교체설에 회사 측 '검토한 바 없다'" → 부정어가 있지만 실제로는 소문 부인이라 중립에 가깝다.

그래서 서로 다른 약점을 가진 모델을 함께 쓰고, 각자의 강점만 골라 쓰도록 결합한다.

### 사용하는 모델 4개
| 역할 | 모델 | 구조와 뿌리 | 하는 일 | 이 모델을 고른 근거 |
| --- | --- | --- | --- | --- |
| 감성 전문가 1 | snunlp/KR-FinBert-SC | BERT, KR-BERT에서 출발 | 문장 → 부정/중립/긍정 | 금융 뉴스 44만 건·리포트 1.1만 건으로 추가 학습. 모델 카드 정확도 0.963 |
| 감성 전문가 2 | kwoncho/KoFinBERT | RoBERTa, ko-sroberta에서 출발 | 문장 → 부정/중립/긍정 | 기업 뉴스 감성 연구(회계학연구 2022) 기반. 학습 자료가 달라 전문가 1과 오류가 겹치지 않을 가능성 |
| 대상 확인 담당 | Huffon/klue-roberta-base-nli | RoBERTa, klue/roberta-base에서 출발 | (문장, 가설) → 맞다/모르겠다/틀리다 | **대상을 직접 지정**할 수 있는 유일한 모델. "B사에 부정적인 소식이다"가 맞는지 묻는다 |
| 중요도 담당 | kwoncho/ko-sroberta-multitask-informative | RoBERTa, ko-sroberta에서 출발 | 문장 → 중요/비중요 | KoFinBERT 모델 카드가 "중요하지 않은 뉴스는 부정확할 수 있다"며 함께 쓰라고 권함 |

### 결합 방식: 가중치 합성이 아니라 출력 합성
```text
                ┌ 감성 전문가 1 ─ 부정·중립·긍정 확률 ┐
문장 + 대상 ──► ├ 감성 전문가 2 ─ 부정·중립·긍정 확률 ┤
 (대상 구절)    ├ 대상 확인 담당 ─ "좋은 소식/나쁜 소식" 맞다 확률 ┼─► 위원장(보정 층) ─► 판정 + 확신도
                ├ 중요도 담당 ─ 중요 확률            ┤    (로지스틱 회귀)
                └ 금융 어휘 규칙 ─ '적자 축소'=긍정 등 ┘
```
- **출력 합성**: 각 모델의 답(확률)을 받아 섞는다. 구조가 달라도 된다.
- **가중치 합성(model merging)**: 모델 내부 숫자를 직접 섞어 새 모델을 만든다. **뿌리가 같은 모델끼리만** 가능하다.
  - 감성 전문가 1(BERT)과 2(RoBERTa)는 뿌리·단어 사전(2만 개 vs 3.2만 개)이 달라 섞을 수 없다.
  - 감성 전문가 2와 중요도 담당은 둘 다 ko-sroberta에서 나왔다. 그래서 5-3절에서 **실험으로만** 섞어 본다.
- **위원장(보정 층)** 은 "어느 전문가 말을 얼마나 믿을지"만 튜닝용 데이터에서 배운다. 전문가(언어모델)는 바뀌지 않는다.

### 비유와 실제의 차이
"위원회"는 이해를 돕는 비유다. 실제로는 확률 숫자를 입력으로 받는 수식 하나다. 위원장이 문장을 직접 읽지는 않는다.

## 1. 환경 설정
- 실행 환경: 이 폴더의 가상환경 `.venv-sentiment` (커널 이름 `kic-sentiment`).
- 수업 환경 `ml-dl-nlp`에는 PyTorch·transformers가 없고, 설치하면 TensorFlow 실습이 깨질 수 있어 따로 만들었다.

In [1]:
import os, re, json, time, hashlib, random, unicodedata   # 표준 라이브러리: 파일·정규식·해시 등
from itertools import product                               # 격자 탐색(grid search)에 쓴다

import numpy as np                                          # 확률 배열 계산
import pandas as pd                                         # 표 데이터 처리 (수업 ch11)
import matplotlib.pyplot as plt                             # 그래프 (수업 ch12)
import requests                                             # 데이터 내려받기 (수업 ch14)
import torch                                                # 언어모델 실행 엔진(추론만, 학습 없음)
from transformers import AutoTokenizer, AutoModelForSequenceClassification   # 허깅페이스 모델 불러오기 (수업 07_LLM ch1)
from sklearn.model_selection import GroupShuffleSplit, GroupKFold            # 기업 단위 분할
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)                         # 성능 지표 (수업 08 3장 6절)
from sklearn.linear_model import LogisticRegression                          # 보정 층(8절)

SEED = 42                                   # 결과를 다시 돌려도 같게 만드는 난수 씨앗
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() - 1))   # CPU 코어를 최대한 써서 추론 속도를 올린다

plt.rcParams['font.family'] = 'Malgun Gothic'        # 그래프 한글 글꼴(Windows)
plt.rcParams['axes.unicode_minus'] = False           # 음수 기호 깨짐 방지
pd.set_option('display.max_colwidth', 80)            # 표에서 문장을 80자까지 보여 준다

DATA_DIR, OUT_DIR = 'data', 'outputs'                # 내려받은 자료와 결과 파일을 둘 폴더
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LABELS = ['negative', 'neutral', 'positive']         # 모든 확률 배열은 이 순서(부정·중립·긍정)로 맞춘다
KO = {'negative': '부정', 'neutral': '중립', 'positive': '긍정'}
print('torch', torch.__version__, '| CPU 스레드', torch.get_num_threads())

E:\AI_융복합수업\2차_팀프로젝트\지배구조_경영진 분석\.venv-sentiment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cpu | CPU 스레드 7


## 2. 평가 데이터 불러오기: KorFin-ASC
한 행은 (문장 `SRC`, 대상 `ASPECT`, 극성 `SENTIMENT`, 유형 `TYPE`)이다. 같은 문장이라도 대상이 다르면 극성이 다를 수 있다.
예: "A사 흑자전환, B사 적자 지속" → A사 긍정, B사 부정. 이것이 **대상 기반 감성**이다.

In [2]:
URL = ('https://huggingface.co/datasets/amphora/korfin-asc/resolve/main/'
       'data/train-00000-of-00001-44ba6365e50c2c0a.parquet')   # 허깅페이스에 공개된 원본 파일
PATH = os.path.join(DATA_DIR, 'korfin_asc.parquet')
if not os.path.exists(PATH):                         # 이미 받아 두었으면 다시 받지 않는다
    with open(PATH, 'wb') as f:
        f.write(requests.get(URL, timeout=60).content)

raw = pd.read_parquet(PATH)                           # parquet: 표를 압축 저장하는 형식
print(raw.shape)
print(raw.columns.tolist())


(8818, 5)
['SID', 'TYPE', 'SRC', 'ASPECT', 'SENTIMENT']


In [3]:
# 열 이름이 대문자·소문자로 섞여 있어도 쓸 수 있게 소문자 기준으로 찾는다
cols = {c.lower(): c for c in raw.columns}
df = pd.DataFrame({
    'sid':    raw[cols['sid']].astype(str).str.split('_').str[0] if 'sid' in cols else raw.index.astype(str),  # 문장 번호(같은 문장의 대상들은 번호가 같다)
    'text':   raw[cols['src'] if 'src' in cols else cols['title']].astype(str),   # 문장
    'aspect': raw[cols['aspect']].astype(str),        # 대상(주로 기업명)
    'label':  raw[cols['sentiment']].astype(str).str.lower(),   # 정답 극성
    'type':   raw[cols['type']].astype(str) if 'type' in cols else 'unknown',  # 한 문장 속 대상이 하나인지 여럿인지
})
# 전체 라벨 분포를 먼저 출력하지 않는다. dev만 분할 후 요약한다.

## 3. 정제와 기업 단위 분할
### 정제
- 전각 문자(＋, Ａ)를 반각으로 바꾼다(유니코드 NFKC 정규화). 예: "LGU＋" → "LGU+"
- 라벨이 3종이 아닌 행, 완전히 같은 (문장, 대상) 중복을 뺀다.

### 분할: 왜 기업 단위인가
같은 기업 문장이 dev와 test에 섞이면, dev에서 맞춘 규칙이 그 기업 표현을 외워서 test 점수가 부풀 수 있다.
그래서 **기업(aspect)이 겹치지 않게** dev 40%, test 60%로 나눈다. test는 "처음 보는 기업" 성능이다.
또 한 문장에 대상이 여럿이면 같은 문장이 양쪽에 들어갈 수 있다. 그런 문장은 test에서 뺀다(`excluded`).

In [4]:
def norm(s):
    # 전각→반각, 연속 공백→한 칸, 앞뒤 공백 제거
    s = unicodedata.normalize('NFKC', str(s))
    return re.sub(r'\s+', ' ', s).strip()

def key(s):
    # 대상 찾기용 비교 키: 공백·마침표·괄호·'(주)' 등을 지우고 소문자로
    s = norm(s).lower()
    s = re.sub(r'\(주\)|㈜|주식회사', '', s)
    return re.sub(r'[\s\.\(\)\[\]·\-_]', '', s)

df['text'] = df['text'].map(norm)
df['aspect'] = df['aspect'].map(norm)
df = df[df['label'].isin(LABELS)].drop_duplicates(['text', 'aspect']).reset_index(drop=True)
df['y'] = df['label'].map({l: i for i, l in enumerate(LABELS)})          # 정답을 숫자(0,1,2)로
df['group'] = df['aspect'].map(key)                                       # 분할 기준: 기업 키

gss = GroupShuffleSplit(n_splits=1, test_size=0.6, random_state=SEED)     # 기업 단위 무작위 분할
dev_idx, test_idx = next(gss.split(df, groups=df['group']))
df['split'] = 'test'
df.loc[dev_idx, 'split'] = 'dev'

# 한 문장에 대상이 여럿이면 같은 문장이 dev와 test에 함께 들어갈 수 있다 → test에서 그런 문장을 뺀다
dup_sid = set(df.loc[df.split == 'dev', 'sid']) & set(df.loc[df.split == 'test', 'sid'])
print('dev와 문장이 겹쳐 test에서 뺀 행:', int((df.split.eq('test') & df.sid.isin(dup_sid)).sum()))
df.loc[df.split.eq('test') & df.sid.isin(dup_sid), 'split'] = 'excluded'
overlap = set(df.loc[df.split == 'dev', 'group']) & set(df.loc[df.split == 'test', 'group'])
print('dev/test 기업 겹침:', len(overlap), '개 (0이어야 정상)')
print(df.groupby('split').agg(문장수=('text', 'size'), 기업수=('group', 'nunique')))
print('dev 라벨 분포만:'); print(df.loc[df.split == 'dev', 'label'].value_counts(normalize=True).round(3))

dev와 문장이 겹쳐 test에서 뺀 행: 1453
dev/test 기업 겹침: 0 개 (0이어야 정상)
           문장수   기업수
split               
dev       3331  1354
excluded  1453   879
test      4034  1518
dev 라벨 분포만:
label
positive    0.349
negative    0.345
neutral     0.306
Name: proportion, dtype: float64


## 4. 대상 구절 자르기 (대상 기반 입력 만들기)
공개 모델은 **문장 전체**의 감성을 낸다. 한 문장에 기업이 여럿이면 다른 기업의 감성이 섞인다.
그래서 문장을 구절로 나누고 **대상이 들어 있는 구절만** 모델에 넣는 입력을 따로 만든다.

### 1차 방식의 문제 (쉼표 기준, 2026-10-27 1차 실행)
1차에서는 쉼표·가운뎃점·말줄임마다 끊었다. 그러자 **사건 내용이 잘려 나갔다.**

| 문장 | 1차에서 잘린 구절 | 빠진 내용 |
| --- | --- | --- |
| A전자 최대주주, 보유 지분 일부 블록딜로 매각 | "A전자 최대주주" | 블록딜로 매각 |
| B화학, 적자 폭 축소…3분기 흑자전환 기대 | "B화학" | 적자 축소, 흑자전환 |

숫자도 같은 방향이었다. KoFinBERT는 문장 전체(dev 0.678)가 1차 구절(0.640)보다 높았다.

### 2차 방식 (대비 연결어 기준)
- **대비 연결어**(`~지만`, `그러나`, `하지만`, `반면`, `~와 달리`, `~에도 불구`)와 **문장 끝**(`~다.`)에서만 끊는다.
- 쉼표·말줄임에서는 끊지 않는다. 한 주어의 사건 설명은 보통 쉼표로 이어지기 때문이다.
- 예: "D제약 영업이익 감소**했지만** | E바이오는 사상 최대" → 둘로 나뉜다. "A전자 최대주주, 블록딜로 매각" → 그대로 하나다.
- 대상이 든 조각을 모두 이어 붙인다. 못 찾으면 문장 전체를 쓴다.

### 모델 입력 4종 비교
① 문장 전체 ② 2차 대상 구절 ③ 2차 대상 구절 + 대상 이름(문장쌍) ④ 1차 쉼표 구절(비교용)
어느 입력이 나은지는 dev 점수로 정한다.

In [5]:
SEP = re.compile(r'\s*(?:,|·|…|\.\.\.|\||/|;|\s-\s|↑|↓)\s*')          # 1차: 쉼표 등 구절 경계 기호
CONTRAST = re.compile(r'(지만|그러나|하지만|반면|와 달리|과 달리|에도 불구)')   # 대비 연결어
SENT_END = re.compile(r'(다\.)\s+')                                     # 2차: 문장 끝 '~다.' 뒤 공백

def pick(parts, text, aspect):
    # 조각 목록에서 대상이 든 조각만 이어 붙인다. 못 찾거나 조각이 하나면 문장 전체
    k = key(aspect)
    hit = [c for c in parts if k and k in key(c)]               # 대상 키가 들어 있는 조각
    if not hit and len(k) >= 3:                                 # 못 찾으면 앞 3글자로 한 번 더(예: '삼성전자' → '삼성전')
        hit = [c for c in parts if k[:3] in key(c)]
    if not hit or len(parts) == 1:
        return text, False
    return ' '.join(hit), True

def clauses(text):
    # 1차(비교용): 대비 연결어 뒤에서 끊고, 쉼표 등 경계 기호로 다시 끊는다
    parts = []
    for chunk in CONTRAST.sub(r'\1⏎', text).split('⏎'):
        parts += [p for p in SEP.split(chunk) if p and p.strip()]
    return parts or [text]

def segments(text):
    # 2차: 대비 연결어 뒤와 문장 끝에서만 끊는다. 쉼표는 끊지 않는다
    t = SENT_END.sub(r'\1⏎', CONTRAST.sub(r'\1⏎', text))
    return [p.strip() for p in t.split('⏎') if p.strip()] or [text]

def comma_clause(text, aspect):
    return pick(clauses(text), text, aspect)

def target_clause(text, aspect):
    return pick(segments(text), text, aspect)

tc = df.apply(lambda r: target_clause(r['text'], r['aspect']), axis=1)
df['clause'] = [t for t, _ in tc]                     # 2차 대상 구절 (이후 모든 단계의 기본 입력)
df['clause_cut'] = [c for _, c in tc]
df['cclause'] = [comma_clause(t, a)[0] for t, a in zip(df['text'], df['aspect'])]   # 1차 쉼표 구절(비교용)
df['aspect_found'] = df.apply(lambda r: key(r['aspect']) in key(r['text']), axis=1)
print('대상 이름이 문장에 그대로 있는 비율:', round(df['aspect_found'].mean(), 3))
print('2차 방식으로 잘라 낸 비율:', round(df['clause_cut'].mean(), 3),
      '| 1차 방식으로 잘라 낸 비율:', round((df['cclause'] != df['text']).mean(), 3))
print('평균 글자 수 → 문장 전체', int(df.text.str.len().mean()), '| 2차 구절', int(df.clause.str.len().mean()),
      '| 1차 구절', int(df.cclause.str.len().mean()))
df.loc[(df.split == 'dev') & df.clause_cut, ['text', 'aspect', 'clause', 'cclause', 'label']].head(6)

대상 이름이 문장에 그대로 있는 비율: 0.994
2차 방식으로 잘라 낸 비율: 0.146 | 1차 방식으로 잘라 낸 비율: 0.635
평균 글자 수 → 문장 전체 75 | 2차 구절 69 | 1차 구절 46


,text,aspect,clause,cclause,label
25,금융위기 이후 현대차그룹의 가파른 성장으로부터 수혜를 입는 부품업체들 주가도 동반 초강세를 보였다. 하지만 현대차그룹의 글로벌 판매대수 ...,현대차그룹,금융위기 이후 현대차그룹의 가파른 성장으로부터 수혜를 입는 부품업체들 주가도 동반 초강세를 보였다. 현대차그룹의 글로벌 판매대수 증가율은...,금융위기 이후 현대차그룹의 가파른 성장으로부터 수혜를 입는 부품업체들 주가도 동반 초강세를 보였다. 하지만 현대차그룹의 글로벌 판매대수...,negative
39,"SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구하고 5월에 마케팅 경쟁이 심화되면서 기대에 못 미치는 5,821억원(YoY +5...",SK텔레콤,SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구,SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구,negative
40,"SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구하고 5월에 마케팅 경쟁이 심화되면서 기대에 못 미치는 5,821억원(YoY +5...",방통위,SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구,SK텔레콤의 영업이익은 방통위의 마케팅비용 규제에도 불구,neutral
44,"부산은행이 경남은행을 인수하더라도 자산총액은 55조원 수준으로 여전히 시중은행 대비 영세한 규모인 반면, 인수자금 상환을 위해 배당여력은...",경남은행,부산은행이 경남은행을 인수하더라도 자산총액은 55조원 수준으로 여전히 시중은행 대비 영세한 규모인 반면,부산은행이 경남은행을 인수하더라도 자산총액은 55조원 수준으로 여전히 시중은행 대비 영세한 규모인 반면,neutral
47,"자회사 별로는 일회성 손실과 기업구조조정 대손비용이 발생한 신한은행과 신한금융투자의 이익 감소 폭이 컸던 반면, 신한카드와 신한생명은 견...",신한카드,", 신한카드와 신한생명은 견조한 실적을 기록하였다.",신한카드와 신한생명은 견조한 실적을 기록하였다.,positive
48,"자회사 별로는 일회성 손실과 기업구조조정 대손비용이 발생한 신한은행과 신한금융투자의 이익 감소 폭이 컸던 반면, 신한카드와 신한생명은 견...",신한생명,", 신한카드와 신한생명은 견조한 실적을 기록하였다.",신한카드와 신한생명은 견조한 실적을 기록하였다.,positive


## 5. 공개 모델로 기준선 측정
- 두 모델을 **불러와서 추론만** 한다. 학습(가중치 변경)은 하지 않는다.
- 모델마다 라벨 순서가 다를 수 있어 `id2label`을 읽어 부정·중립·긍정 순서로 맞춘다.
- 같은 입력은 한 번만 계산하고 결과를 `data/`에 저장해 두었다가 다시 쓴다(캐시).

In [6]:
MODELS = {'finbert': 'snunlp/KR-FinBert-SC', 'kofin': 'kwoncho/KoFinBERT'}      # 감성 전문가 2명
AUX = {'nli': 'Huffon/klue-roberta-base-nli',                                    # 대상 확인 담당
       'info': 'kwoncho/ko-sroberta-multitask-informative'}                      # 중요도 담당

def load(name, labels=LABELS):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name).eval()   # eval(): 추론 모드
    id2 = {v.lower(): int(k) for k, v in mdl.config.id2label.items()}
    order = [id2[l] for l in labels]                    # 원하는 라벨 순서로 열을 재배열할 위치
    return tok, mdl, order

@torch.no_grad()                                     # 기울기 계산 끔 = 학습하지 않음
def predict(tag, texts, pairs=None, bs=64, bundle=None):
    # 입력 목록의 해시로 캐시 파일 이름을 정한다
    h = hashlib.md5(('|'.join(texts) + '||' + '|'.join(pairs or [])).encode()).hexdigest()[:10]
    path = os.path.join(DATA_DIR, f'pred_{tag}_{h}.npy')
    if os.path.exists(path):
        return np.load(path)
    tok, mdl, order = bundle or BUNDLE[tag.split('_')[0]]
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], pairs[i:i+bs] if pairs else None,
                  padding=True, truncation=True, max_length=192, return_tensors='pt')
        enc.pop('token_type_ids', None) if 'roberta' in mdl.config.model_type else None
        p = torch.softmax(mdl(**enc).logits, -1).numpy()
        out.append(p[:, order])                       # 부정·중립·긍정 순서로 재배열
    arr = np.vstack(out)
    np.save(path, arr)
    return arr

t0 = time.time()
BUNDLE = {k: load(v) for k, v in MODELS.items()}
print('모델 불러오기', round(time.time() - t0, 1), '초')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9380.93it/s]

모델 불러오기 5.2 초


In [7]:
# 입력 4종 × 모델 2개 = 확률 8세트. 중복 문장은 한 번만 계산한 뒤 행에 다시 펼친다.
# 캐시 이름(tag)은 입력 문장이 같으면 1차 실행 결과를 그대로 다시 쓴다(1차 쉼표 구절은 재계산 없음).
def run(tag, col, use_pair=False):
    if use_pair:
        uniq = df[[col, 'aspect']].drop_duplicates().reset_index(drop=True)
        p = predict(tag, uniq[col].tolist(), uniq['aspect'].tolist())
        m = {(a, b): p[i] for i, (a, b) in enumerate(zip(uniq[col], uniq['aspect']))}
        return np.vstack([m[(a, b)] for a, b in zip(df[col], df['aspect'])])
    uniq = df[col].drop_duplicates().tolist()
    p = predict(tag, uniq)
    m = dict(zip(uniq, p))
    return np.vstack([m[t] for t in df[col]])

P = {}
t0 = time.time()
for mk in MODELS:
    P[f'{mk}_full']   = run(f'{mk}_full', 'text')            # ① 문장 전체
    P[f'{mk}_clause'] = run(f'{mk}_seg', 'clause')           # ② 2차 대상 구절
    P[f'{mk}_pair']   = run(f'{mk}_segpair', 'clause', True) # ③ 2차 대상 구절 + 대상 이름
    P[f'{mk}_comma']  = run(f'{mk}_clause', 'cclause')       # ④ 1차 쉼표 구절(비교용, 1차 캐시 재사용)
    print(mk, '완료', round(time.time() - t0, 1), '초')

finbert 완료 0.2 초
kofin 완료 0.4 초


In [8]:
def score(y, pred):
    # 정확도와 macro-F1(세 라벨을 똑같이 중요하게 본 F1 평균)을 함께 본다
    return {'accuracy': round(accuracy_score(y, pred), 4),
            'macro_F1': round(f1_score(y, pred, average='macro'), 4)}

dev = (df.split == 'dev').values
test = (df.split == 'test').values
base = pd.DataFrame({k: score(df.y[dev], v[dev].argmax(1)) for k, v in P.items()}).T
print('[dev] 기준선: 모델 × 입력 방식')
base.sort_values('macro_F1', ascending=False)

[dev] 기준선: 모델 × 입력 방식


,accuracy,macro_F1
kofin_pair,0.6971,0.6869
kofin_clause,0.6971,0.6859
kofin_full,0.6896,0.6784
kofin_comma,0.6391,0.6400
finbert_clause,0.6103,0.5958
finbert_pair,0.6073,0.5937
finbert_comma,0.5986,0.5912
finbert_full,0.5974,0.5829


## 5-2. 대상 확인 담당(NLI)과 중요도 담당 추가

### 대상 확인 담당: 자연어 추론(NLI)을 감성 판정에 쓰는 원리
NLI 모델은 원래 "문장(전제)이 가설을 뒷받침하는가"를 맞다·모르겠다·틀리다로 판정한다.
이 성질을 이용해 **대상을 지정한 질문**을 두 번 던진다.

| 전제(대상 구절) | 가설 | 기대 답 |
| --- | --- | --- |
| "B사 적자 지속" | "B사에 긍정적인 소식이다." | 틀리다 |
| "B사 적자 지속" | "B사에 부정적인 소식이다." | 맞다 |

- 두 질문의 "맞다" 확률을 각각 긍정·부정 점수로 쓰고, "모르겠다" 확률은 중립 점수로 쓴다.
- 학습 없이 쓰는 **제로샷** 방식이다. 감성 데이터로 학습한 모델이 아니므로 혼자서는 감성 전문가보다 약할 수 있다.
- 대신 "대상"을 볼 수 있다는 다른 강점이 있다. 그래서 단독으로 쓰지 않고 위원장(8절)의 입력으로 쓴다.

### 중요도 담당
- 문장이 기업에 중요한 뉴스인지 확률을 낸다. 라벨 이름이 '0', '1'로만 공개되어 있어 어느 쪽이 '중요'인지 모델 카드로 확인하지 못했다.
- 그래서 판정 규칙에 직접 넣지 않는다. 위원장(로지스틱 회귀)이 이 값을 어떻게 쓸지 dev에서 스스로 정한다. 방향을 몰라도 영향은 배울 수 있다.

In [9]:
import copy                                                        # 5-3절 모델 복사용
BUNDLE['nli'] = load(AUX['nli'], ['entailment', 'neutral', 'contradiction'])   # 맞다·모르겠다·틀리다 순서
BUNDLE['info'] = load(AUX['info'], ['0', '1'])
HYP = {'pos': '{a}에 긍정적인 소식이다.', 'neg': '{a}에 부정적인 소식이다.'}    # 대상을 넣은 가설 문장

pairs_df = df[['clause', 'aspect']].drop_duplicates().reset_index(drop=True)
t0 = time.time()
nli = {h: predict(f'nli_{h}', pairs_df['clause'].tolist(),
                  [HYP[h].format(a=a) for a in pairs_df['aspect']], bundle=BUNDLE['nli'])
       for h in HYP}                                               # 가설 2개 × (구절, 대상) 쌍
m = {(c, a): np.concatenate([nli['pos'][i], nli['neg'][i]])
     for i, (c, a) in enumerate(zip(pairs_df['clause'], pairs_df['aspect']))}
NLI = np.vstack([m[(c, a)] for c, a in zip(df['clause'], df['aspect'])])   # 6열: 긍정가설(맞다·모름·틀림), 부정가설(같음)

def nli_to3(x):
    # NLI 6열 → 부정·중립·긍정 3열 확률. 부정=부정가설 '맞다', 긍정=긍정가설 '맞다', 중립=두 '모르겠다'의 평균
    s = np.c_[x[:, 3], (x[:, 1] + x[:, 4]) / 2, x[:, 0]]
    return s / s.sum(1, keepdims=True)

P['nli_zero'] = nli_to3(NLI)

uniq = df['clause'].drop_duplicates().tolist()
pi = predict('info_clause', uniq, bundle=BUNDLE['info'])
INFO = np.vstack([dict(zip(uniq, pi))[t] for t in df['clause']])[:, 1:2]   # '1' 라벨 확률 한 열
EXTRA = np.hstack([NLI, INFO])                                       # 위원장 입력에 더할 7열
print('NLI·중요도 추론', round(time.time() - t0, 1), '초')
print('[dev] NLI 대상 질문 단독(제로샷):', score(df.y[dev], P['nli_zero'][dev].argmax(1)))
print('[dev] 중요도 확률 평균(라벨별):'); print(pd.Series(INFO[dev, 0]).groupby(df.label[dev].values).mean().round(3))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8065.12it/s]


NLI·중요도 추론 19.2 초
[dev] NLI 대상 질문 단독(제로샷): {'accuracy': 0.4194, 'macro_F1': 0.3668}
[dev] 중요도 확률 평균(라벨별):
negative    0.657
neutral     0.533
positive    0.641
dtype: float32


## 5-3. 가중치 합성 실험 (뿌리가 같은 두 모델만)

### 무엇을 하나
KoFinBERT(감성)와 중요도 담당은 둘 다 `jhgan/ko-sroberta-multitask`에서 출발했다. 구조와 단어 사전이 같다.
그래서 **과제 벡터 더하기(task arithmetic)** 를 시험한다.

```text
합성 모델 = KoFinBERT + λ × (중요도 모델 − 공통 뿌리 모델)
```
- `(중요도 모델 − 공통 뿌리)`는 "중요도를 배우면서 바뀐 부분"이다. 이것을 KoFinBERT에 λ만큼 더한다.
- 판정 머리(분류층)는 KoFinBERT 것을 그대로 쓴다. 본문(인코더)만 섞는다.
- 학습은 없다. 숫자를 더하기만 한다.

### 채택 기준
- λ = 0은 원래 KoFinBERT와 같아야 한다(구현 점검).
- dev macro-F1이 원래보다 **0.005 넘게** 오를 때만 채택해 위원장 입력에 더한다. 오르지 않으면 버리고 결과만 기록한다.
- 두 모델이 서로 다른 일을 배웠기 때문에 오히려 떨어질 수 있다. 그래서 "실험"이다.

In [10]:
from transformers import AutoModel
base_enc = AutoModel.from_pretrained('jhgan/ko-sroberta-multitask')        # 공통 뿌리(기준점)
k_tok, k_mdl, k_order = BUNDLE['kofin']
sd_k, sd_i, sd_b = k_mdl.state_dict(), BUNDLE['info'][1].state_dict(), base_enc.state_dict()

def merged_model(lam):
    new, hit = {}, 0
    for n, w in sd_k.items():
        bn = n[len('roberta.'):] if n.startswith('roberta.') else None    # 뿌리 모델은 'roberta.' 접두어가 없다
        if bn and bn in sd_b and n in sd_i and sd_b[bn].shape == w.shape and w.dtype.is_floating_point:
            new[n] = w + lam * (sd_i[n] - sd_b[bn]); hit += 1             # 본문(인코더)만 섞는다
        else:
            new[n] = w                                                    # 분류 머리 등은 그대로
    m = copy.deepcopy(k_mdl); m.load_state_dict(new)
    return m.eval(), hit

dev_clauses = df.loc[dev, 'clause'].drop_duplicates().tolist()
merge_res = {}
for lam in [0.0, 0.25, 0.5]:
    mm, hit = merged_model(lam)
    p = predict(f'merged{int(lam*100)}_clause', dev_clauses, bundle=(k_tok, mm, k_order))
    mp = dict(zip(dev_clauses, p))
    pr = np.vstack([mp[t] for t in df.loc[dev, 'clause']]).argmax(1)
    merge_res[lam] = f1_score(df.y[dev], pr, average='macro')
    print(f'λ={lam}: 섞은 가중치 텐서 {hit}개, dev macro-F1 {merge_res[lam]:.4f}')

print('점검: λ=0 결과가 KoFinBERT(대상 구절) dev 점수와 같은가 →',
      round(merge_res[0.0], 4), 'vs', base.loc['kofin_clause', 'macro_F1'])
BEST_LAM = max(merge_res, key=merge_res.get)
MERGE_ADOPTED = BEST_LAM > 0 and merge_res[BEST_LAM] - merge_res[0.0] > 0.005
if MERGE_ADOPTED:
    BUNDLE['merged'] = (k_tok, merged_model(BEST_LAM)[0], k_order)
    P['merged_clause'] = run('merged_clause', 'clause')
    print(f'채택: λ={BEST_LAM} 합성 모델을 위원장 입력에 추가')
else:
    print('미채택: 합성이 dev 성능을 올리지 못해 버린다(결과만 기록)')
del base_enc

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 533.64it/s]


λ=0.0: 섞은 가중치 텐서 197개, dev macro-F1 0.6859
λ=0.25: 섞은 가중치 텐서 197개, dev macro-F1 0.6813
λ=0.5: 섞은 가중치 텐서 197개, dev macro-F1 0.6668
점검: λ=0 결과가 KoFinBERT(대상 구절) dev 점수와 같은가 → 0.6859 vs 0.6859
미채택: 합성이 dev 성능을 올리지 못해 버린다(결과만 기록)


## 6. 금융 어휘 규칙: 애매한 표현·사실 서술 다루기
모델이 자주 헷갈리는 표현을 규칙으로 보완한다. 규칙은 **점수(특징)** 로만 만들고, 판정은 9절에서 모델 확률과 합친다.

| 규칙 | 예 | 처리 |
| --- | --- | --- |
| 사실 서술의 방향 | "영업익 30% 증가", "매출 감소" | 증가·감소 같은 방향어로 극성을 매긴다 |
| 나쁜 것의 감소 | "적자 축소", "손실 줄어" | 나쁜 명사 + 감소어 = **긍정** |
| 나쁜 것의 증가 | "적자 확대", "부채 늘어" | 나쁜 명사 + 증가어 = **부정** |
| 부정어 | "흑자전환 못해", "수주 무산" | 긍정어 뒤 부정어 = 뒤집기 |
| 헤지(추측) | "회복 전망", "인상 가능성" | 확신을 낮추는 표시. 중립 쪽으로 기울 수 있다 |
| 대비 | "매출은 늘었지만 이익은 줄어" | 4절에서 대상 구절로 이미 분리 |

규칙 어휘는 금융 일반 지식으로 정했다. test 문장을 보고 고치지 않았다.

In [11]:
POS_W = ['증가', '급증', '상승', '급등', '호조', '호실적', '개선', '흑자', '최대', '최고', '사상 최', '돌파',
         '수주', '상향', '성장', '회복', '반등', '강세', '수혜', '신고가', '순항', '확대', '인상', '기대',
         '흥행', '선방', '어닝 서프라이즈', '서프라이즈', '견조', '안정', '청신호', '훈풍', '최초', '1위', '선정',
         '승인', '체결', '출시', '본격화', '정상화', '턴어라운드', '저평가', '매수']
NEG_W = ['감소', '급감', '하락', '급락', '부진', '악화', '적자', '손실', '하향', '약세', '우려', '리스크',
         '위기', '소송', '횡령', '배임', '구속', '기소', '파업', '중단', '리콜', '제재', '과징금', '부도',
         '상장폐지', '하회', '쇼크', '둔화', '빨간불', '경고', '압박', '부담', '논란', '의혹', '피소',
         '불확실', '침체', '역성장', '고전', '타격', '차질', '지연', '철수', '매도', '고평가', '취소']
BAD_N = ['적자', '손실', '부채', '리스크', '우려', '비용', '부담', '적자폭', '손실폭', '불확실성', '재고']
DOWN = ['감소', '축소', '줄', '개선', '해소', '탈피', '벗어', '완화', '덜', '감축', '하락']
UP = ['확대', '증가', '늘', '지속', '심화', '가중', '커', '불어']
NEGATION = ['않', '못', '없', '무산', '철회', '불발', '실패', '아니', '미달']
HEDGE = ['전망', '예상', '관측', '가능성', '검토', '추진', '계획', '예정', '?', '설 ', '촉각', '주목', '변수',
         '갈림길', '기로', '불투명', '관건', '여부']

def has(text, words):
    return [w for w in words if w in text]

def lex_features(t):
    # 대상 구절 t 하나에서 규칙 점수를 만든다
    pos, neg = has(t, POS_W), has(t, NEG_W)
    # 나쁜 명사 뒤 6글자 안에 감소어 → 긍정, 증가어 → 부정 (예: '적자 폭 축소')
    bad_down = len(re.findall(r'(%s).{0,6}(%s)' % ('|'.join(BAD_N), '|'.join(DOWN)), t))
    bad_up = len(re.findall(r'(%s).{0,6}(%s)' % ('|'.join(BAD_N), '|'.join(UP)), t))
    # 긍정어 뒤 8글자 안에 부정어 → 긍정을 뒤집는다 (예: '흑자전환 실패')
    neg_flip = len(re.findall(r'(%s).{0,8}(%s)' % ('|'.join(map(re.escape, POS_W)), '|'.join(NEGATION)), t))
    hedge = len(has(t, HEDGE))
    numeric = int(bool(re.search(r'\d', t)))                      # 숫자가 있으면 사실 서술일 가능성
    # 최종 규칙 점수: 긍정어 - 부정어 + 2×(나쁜 것 감소) - 2×(나쁜 것 증가) - 2×(부정어 뒤집기)
    # '적자 축소'는 '적자'(부정어)도 세지므로 2배로 보정한다
    s = len(pos) - len(neg) + 2 * bad_down - 2 * bad_up - 2 * neg_flip
    return {'lex': s, 'n_pos': len(pos), 'n_neg': len(neg), 'bad_down': bad_down, 'bad_up': bad_up,
            'neg_flip': neg_flip, 'hedge': hedge, 'numeric': numeric}

LX = pd.DataFrame([lex_features(t) for t in df['clause']])
df = pd.concat([df.drop(columns=[c for c in LX.columns if c in df.columns]), LX], axis=1)

# 규칙만으로 판정했을 때 성능(참고): 점수>0 긍정, <0 부정, 0 중립
lex_pred = np.where(df.lex > 0, 2, np.where(df.lex < 0, 0, 1))
print('[dev] 규칙만:', score(df.y[dev], lex_pred[dev]))
print('[dev] 규칙 점수별 정답 분포'); print(pd.crosstab(np.sign(df.lex[dev]), df.label[dev]))

[dev] 규칙만: {'accuracy': 0.601, 'macro_F1': 0.601}
[dev] 규칙 점수별 정답 분포
label  negative  neutral  positive
lex                               
-1          654      163        63
 0          334      563       314
 1          162      293       785


## 7. 결합 A: 두 모델 + 규칙 (학습 없이 격자 탐색)
1. 각 모델에서 dev 성능이 가장 좋은 입력 방식을 고른다.
2. 두 모델 확률을 가중 평균한다. 가중치 `w`는 0~1 사이에서 찾는다.
3. 규칙 점수를 로그 확률에 더한다. 긍정 쪽은 `+a×점수`, 부정 쪽은 `-a×점수`다.
4. 헤지 표현이 있으면 중립에 `h`를 더한다. 모든 문장의 중립 확률에도 `b`를 더해 기준을 조정한다.
5. `w, a, h, b`는 **dev macro-F1이 가장 높은 값**으로 정한다. 언어모델 가중치는 바꾸지 않는다.

In [12]:
best_input = {mk: base.loc[[f'{mk}_{s}' for s in ['full', 'clause', 'pair', 'comma']], 'macro_F1'].idxmax()
              for mk in MODELS}
print('모델별 최적 입력(dev):', best_input)
PA, PB = P[best_input['finbert']], P[best_input['kofin']]

def combine(w, a, h, b, idx=slice(None)):
    p = w * PA[idx] + (1 - w) * PB[idx]                  # 두 모델 확률 가중 평균
    z = np.log(p + 1e-9)                                 # 로그 확률로 바꿔 규칙 점수를 더한다
    lex = df['lex'].values[idx]
    z[:, 2] += a * lex                                   # 규칙 점수 +면 긍정 쪽으로
    z[:, 0] -= a * lex                                   # 규칙 점수 -면 부정 쪽으로
    z[:, 1] += h * (df['hedge'].values[idx] > 0) + b     # 헤지면 중립 쪽으로, b는 중립 기준 조정
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)                   # 다시 확률로

grid = list(product(np.linspace(0, 1, 11), [0, .1, .2, .3, .5, .8], [0, .2, .4, .8], [-.6, -.3, 0, .3, .6]))
res = []
for w, a, h, b in grid:
    pr = combine(w, a, h, b, dev).argmax(1)
    res.append((f1_score(df.y[dev], pr, average='macro'), w, a, h, b))
res.sort(reverse=True)
F1A, W, A, H, B = res[0]
print(f'최적(dev): w={W:.1f} a={A} h={H} b={B} → dev macro-F1 {F1A:.4f}')
print('상위 5개 조합:'); print(pd.DataFrame(res[:5], columns=['dev_F1', 'w', 'a', 'h', 'b']).round(4))

모델별 최적 입력(dev): {'finbert': 'finbert_clause', 'kofin': 'kofin_pair'}
최적(dev): w=0.3 a=0.2 h=0 b=-0.6 → dev macro-F1 0.6914
상위 5개 조합:
   dev_F1    w    a    h    b
0  0.6914  0.3  0.2  0.0 -0.6
1  0.6912  0.3  0.2  0.2 -0.6
2  0.6910  0.3  0.2  0.8 -0.6
3  0.6909  0.3  0.2  0.4 -0.6
4  0.6905  0.3  0.1  0.0 -0.6


## 8. 결합 B: 보정 층(로지스틱 회귀)
- 이것이 0-2절 그림의 **위원장**이다.
- 입력: 감성 전문가 2명 × 입력 방식 3종의 확률, NLI 대상 질문 확률, 중요도 확률, (채택 시) 합성 모델 확률, 규칙 특징 8개, 문장 유형. 출력: 부정·중립·긍정.
- 언어모델은 그대로 두고, 그 **출력을 어떻게 섞을지만** dev에서 배운다. 파인튜닝이 아니다(수업 08 3장 로지스틱 회귀).
- 과적합을 막으려고 dev 안에서도 기업 단위 5겹 교차검증으로 규제 강도 `C`를 고른다.

In [13]:
FEAT_LEX = ['lex', 'n_pos', 'n_neg', 'bad_down', 'bad_up', 'neg_flip', 'hedge', 'numeric']
def features(idx):
    X = [np.log(P[k][idx] + 1e-9) for k in sorted(P)]               # 모델별 부정·중립·긍정 로그 확률
    X.append(EXTRA[idx])                                           # NLI 원 확률 6열 + 중요도 1열
    X.append(df.loc[idx, FEAT_LEX].values)                         # 규칙 특징
    X.append((df.loc[idx, 'type'].str.contains('multi', case=False)).values[:, None].astype(float))
    X.append(df.loc[idx, 'clause_cut'].values[:, None].astype(float))
    return np.hstack(X)

Xd, yd, gd = features(dev), df.y[dev].values, df.group[dev].values
cv = GroupKFold(n_splits=5)
cv_res = {}
for Cc in [0.03, 0.1, 0.3, 1, 3]:
    f1s = []
    for tr, va in cv.split(Xd, yd, gd):
        lr = LogisticRegression(C=Cc, max_iter=2000, class_weight='balanced').fit(Xd[tr], yd[tr])
        f1s.append(f1_score(yd[va], lr.predict(Xd[va]), average='macro'))
    cv_res[Cc] = np.mean(f1s)
BEST_C = max(cv_res, key=cv_res.get)
print('dev 교차검증 macro-F1:', {k: round(v, 4) for k, v in cv_res.items()}, '→ C =', BEST_C)
LR = LogisticRegression(C=BEST_C, max_iter=2000, class_weight='balanced').fit(Xd, yd)   # dev 전체로 다시 맞춤

dev 교차검증 macro-F1: {0.03: 0.7036, 0.1: 0.7031, 0.3: 0.7036, 1: 0.7024, 3: 0.7035} → C = 0.3


## 9. 2차 기준 모델 선택 (dev만)

여기서는 개발 데이터만으로 2차 기준 결합 방식을 고른다. 원본 KorFin-ASC test와 지배구조 점검 세트의 정답·성능은 4차 어휘와 강도를 고른 뒤 마지막 평가 절에서만 사용한다.

In [14]:
# 기존 기준: 2차 결합 A와 B 중 dev 교차검증 점수가 나은 쪽을 고른다.
dev_A = f1_score(yd, combine(W, A, H, B, dev).argmax(1), average='macro')
dev_B = cv_res[BEST_C]
FINAL_2 = '결합 B: 보정 층(로지스틱)' if dev_B > dev_A else '결합 A: 두 모델 + 규칙'
FINAL = FINAL_2
print(f'dev 비교: A {dev_A:.4f} vs B(교차검증) {dev_B:.4f} → 2차 기준 = {FINAL_2}')

dev 비교: A 0.6914 vs B(교차검증) 0.7036 → 2차 기준 = 결합 B: 보정 층(로지스틱)


## 9-2. 3차: 지배구조 사건 어휘 규칙

### 왜 따로 만드나
- 2차 규칙(6절)은 실적·주가 같은 **일반 금융 어휘**다. '자사주 소각', '블록딜', '오버행 해소' 같은 지배구조 사건은 거의 모른다.
- 지배구조 문장은 감정어 없이 사건만 적는 경우가 많다. 예: "최대주주, 지분 3% 블록딜로 매각". 감성 모델은 이런 문장을 중립으로 보기 쉽다.

### 규칙의 구성
| 묶음 | 뜻 | 예 |
| --- | --- | --- |
| 긍정 사건 | 소액주주에게 유리한 사건 | 자사주 소각, 배당 확대, 무죄 |
| 부정 사건 | 소액주주에게 불리한 사건 | 블록딜, 오버행, 자사주 처분, 경영권 분쟁 |
| 해소 | 부정 사건 뒤 10글자 안 → 긍정 쪽 | 오버행 우려 **해소** |
| 취소 | 긍정 사건 뒤 10글자 안 → 부정 쪽 | 소각 계획 **철회** |
| 부인 | 사건을 부인·해명 → 중립 쪽 | "검토한 바 없다", 매각은 **없다** |

- 2차 일반 어휘(POS_W·NEG_W)에 이미 있는 단어(횡령·배임·구속·기소 등)는 넣지 않았다. 같은 단어를 두 번 세지 않기 위해서다.
- 어휘는 사람이 정한 초안이다. 점검 세트를 보고 고치지 않는다.

In [15]:
# 3차: 지배구조 사건 어휘. 2차 일반 어휘(POS_W·NEG_W)에 있는 단어는 빼서 중복 가중을 막는다
GOV_POS = ['자사주 소각', '자기주식 소각', '주식 소각', '자사주 매입', '자사주 취득', '자기주식 취득',
           '공개매수', '배당 확대', '배당 증액', '배당금 확대', '고배당', '특별배당', '주주환원', '주주친화',
           '무죄', '사면', '복권', '지배구조 개선', '책임경영', '경영권 안정', '지분 매입', '장내 매수']
GOV_NEG = ['블록딜', '지분 매각', '주식 매각', '장내 매도', '대량 매도', '오버행', '물량 출회', '물량 부담',
           '자사주 처분', '자기주식 처분', '자사주 매각', '교환사채', '전환사채', 'CB 발행', '물적분할',
           '쪼개기 상장', '중복 상장', '경영권 분쟁', '경영권 방어', '표 대결', '실형', '압수수색', '검찰 수사',
           '사임', '사퇴', '경질', '배당 축소', '배당 삭감', '무배당', '배당 중단', '배당락', '주주가치 훼손',
           '희석', '소액주주 반발', '꼼수', '일감 몰아주기', '사익편취', '순환출자', '지배구조 낙후', '의견거절']
RESOLVE = ['해소', '해결', '완납', '종결', '종료', '마무리', '봉합', '무죄', '철회', '취하', '기각', '벗어',
           '덜어', '소진', '완화']                       # 부정 사건 뒤에 오면 해소 → 긍정 쪽
CANCEL = ['철회', '무산', '취소', '연기', '보류', '중단', '불발', '백지화', '축소', '번복']   # 긍정 사건 뒤 → 부정 쪽
DENY = ['부인', '사실무근', '검토한 바 없', '계획 없', '계획이 없', '없다고', '루머', '풍문', '해명']  # 부인 → 중립 쪽
NEGATE = ['없', '않', '아니']                         # '매각은 없다'처럼 부정 사건 자체를 부정

def alt(words):
    # 긴 단어부터 찾게 해서 '지분 희석'과 '희석'처럼 겹치는 단어를 두 번 세지 않는다
    return '|'.join(map(re.escape, sorted(set(words), key=len, reverse=True)))

RX_POS, RX_NEG, RX_DENY = re.compile(alt(GOV_POS)), re.compile(alt(GOV_NEG)), re.compile(alt(DENY))
RX_RES = re.compile(f'({alt(GOV_NEG)}).{{0,10}}({alt(RESOLVE)})')      # 부정 사건 + 해소
RX_CAN = re.compile(f'({alt(GOV_POS)}).{{0,10}}({alt(CANCEL)})')       # 긍정 사건 + 취소
RX_NEGATED = re.compile(f'({alt(GOV_NEG)}).{{0,8}}({alt(NEGATE)})')    # 부정 사건 + 없다·않다
# 지배구조 문장 판별: 2차 목록과 같되 '지분법'(회계 용어)은 뺀다
RX_GOVWORD = re.compile(r'대주주|최대주주|지분(?!법)|자사주|자기주식|경영권|대표|회장|이사회|배당|승계|상속|증여|'
                        r'블록딜|오너|지배구조|주총|주주|CEO|임원|사장')

def gov_features(t, _RX_POS=RX_POS, _RX_NEG=RX_NEG, _RX_DENY=RX_DENY, _RX_RES=RX_RES, _RX_CAN=RX_CAN, _RX_NEGATED=RX_NEGATED, _RX_GOVWORD=RX_GOVWORD):
    pos, neg = len(_RX_POS.findall(t)), len(_RX_NEG.findall(t))
    res, can, negated = len(_RX_RES.findall(t)), len(_RX_CAN.findall(t)), len(_RX_NEGATED.findall(t))
    # 점수 = 긍정 사건 - 부정 사건 + 2×해소 - 2×취소 + 1×(부정 사건을 부정)
    # 해소·취소는 원래 사건을 이미 한 번 셌으므로 2배로 뒤집는다. '매각은 없다'는 0으로 되돌리기만 한다
    return {'gov': pos - neg + 2 * res - 2 * can + negated,
            'gov_deny': len(_RX_DENY.findall(t)) + negated,
            'gov_hit': pos + neg,
            'is_gov': bool(_RX_GOVWORD.search(t)) or pos + neg > 0}

GF = pd.DataFrame([gov_features(t) for t in df['clause']])          # 대상 구절 기준(2차 규칙과 같은 입력)
for c in ['gov', 'gov_deny', 'gov_hit']:
    df[c] = GF[c].values
df['is_gov'] = df['text'].map(lambda t: gov_features(t)['is_gov'])  # 문장 판별은 2차처럼 문장 전체 기준
GS, GD = df['gov'].values.astype(float), df['gov_deny'].values.astype(float)

old_gov = df['text'].map(lambda t: any(g in t for g in ['대주주', '최대주주', '지분', '자사주', '경영권', '대표', '회장',
    '이사회', '배당', '승계', '상속', '증여', '블록딜', '오너', '지배구조', '주총', '주주', 'CEO', '임원', '사장']))
print('지배구조 문장 수(dev) → 2차 목록:', int(old_gov[dev].sum()), '| 3차(지분법 제외):', int(df.is_gov[dev].sum()))
act = (GS != 0) | (GD > 0)
print('어휘 규칙이 작동한 문장(dev):', int(act[dev].sum()))
print('[dev] 지배구조 점수 부호별 정답 분포(규칙이 작동한 문장)')
pd.crosstab(np.sign(df.gov[dev & act]), df.label[dev & act])

지배구조 문장 수(dev) → 2차 목록: 214 | 3차(지분법 제외): 184
어휘 규칙이 작동한 문장(dev): 58
[dev] 지배구조 점수 부호별 정답 분포(규칙이 작동한 문장)


label,negative,neutral,positive
gov,,,
-1,14,18,4
0,1,7,1
1,1,6,6


## 9-3. 조정 층과 채택 규칙

### 조정 층
- 최종 방식의 확률을 로그로 바꾼 뒤 지배구조 점수 × 강도 `g`만큼 긍정·부정 쪽으로 민다.
- 부인 표현이 있으면 강도 `d`만큼 중립 쪽으로 민다. 6절 결합 A가 규칙을 더하는 방식과 같다.
- 지배구조 어휘가 없는 문장은 점수가 0이라 확률이 그대로다.

### 채택 규칙 (결과를 보기 전에 정함)
1. 튜닝 자료는 **dev만** 쓴다. 결합 B는 교차검증 확률을 써서 낙관적인 점수를 피한다.
2. dev 지배구조 문장의 macro-F1이 강도 0보다 **0.01 이상** 올라야 한다.
3. dev 전체 macro-F1은 **0.002 넘게 떨어지면 안 된다.**
4. 조건을 만족하는 조합 가운데 최고점과 0.005 안쪽이면 **더 약한 강도**를 고른다.
5. 만족하는 조합이 없으면 채택하지 않는다(g=0, d=0). 그러면 3차 최종은 2차 최종과 같다.
- test와 점검 세트는 강도를 정한 뒤 한 번만 본다.

In [16]:
G_STR, D_STR = 0.0, 0.0                                  # 채택 전 기본값: 조정 없음

def adjust(p, s, dn, g=None, d=None):
    # p: 부정·중립·긍정 확률, s: 지배구조 점수, dn: 부인 개수. g·d를 주지 않으면 채택된 강도를 쓴다
    g = G_STR if g is None else g
    d = D_STR if d is None else d
    z = np.log(np.asarray(p) + 1e-9)
    z[:, 2] += g * s; z[:, 0] -= g * s                    # 점수 +면 긍정 쪽, -면 부정 쪽
    z[:, 1] += d * (np.asarray(dn) > 0)                    # 부인 표현이 있으면 중립 쪽
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def macro(y, p):
    return f1_score(y, p.argmax(1), average='macro')

# dev 확률: 결합 B는 교차검증 확률(낙관 방지), 결합 A는 dev로 고른 조합 그대로
if FINAL.startswith('결합 B'):
    P_DEV = np.zeros((dev.sum(), 3))
    for tr, va in cv.split(Xd, yd, gd):
        P_DEV[va] = LogisticRegression(C=BEST_C, max_iter=2000, class_weight='balanced').fit(Xd[tr], yd[tr]).predict_proba(Xd[va])
else:
    P_DEV = combine(W, A, H, B, dev)

gov_dev = df.loc[dev, 'is_gov'].values
tune_rows = []
for gg, dd in product([0, .25, .5, .75, 1, 1.5, 2, 3], [0, .5, 1, 2]):
    pa = adjust(P_DEV, GS[dev], GD[dev], gg, dd)
    tune_rows.append({'g': gg, 'd': dd, 'dev 전체': macro(yd, pa), 'dev 지배구조': macro(yd[gov_dev], pa[gov_dev])})
tune = pd.DataFrame(tune_rows)
tune['지배구조 변화'] = tune['dev 지배구조'] - tune.loc[0, 'dev 지배구조']   # 0행 = g=0, d=0
tune['전체 변화'] = tune['dev 전체'] - tune.loc[0, 'dev 전체']
tune_ok = tune[(tune['지배구조 변화'] >= 0.01) & (tune['전체 변화'] >= -0.002)]
if len(tune_ok):
    top = tune_ok['dev 지배구조'].max()
    chosen = tune_ok[tune_ok['dev 지배구조'] >= top - 0.005].sort_values(['g', 'd']).iloc[0]
    G_STR, D_STR = float(chosen['g']), float(chosen['d'])
ADOPTED = (G_STR, D_STR) != (0.0, 0.0)
print(f"강도 0일 때 dev → 전체 {tune.loc[0, 'dev 전체']:.4f}, 지배구조 {tune.loc[0, 'dev 지배구조']:.4f} (지배구조 {gov_dev.sum()}문장)")
print('채택' if ADOPTED else '미채택(조건을 넘는 조합 없음)', f'→ g={G_STR}, d={D_STR}')
tune.sort_values('dev 지배구조', ascending=False).head(8).round(4)

강도 0일 때 dev → 전체 0.7042, 지배구조 0.6194 (지배구조 184문장)
채택 → g=0.25, d=0.0


,g,d,dev 전체,dev 지배구조,지배구조 변화,전체 변화
10,0.50,1.0,0.7058,0.6418,0.0224,0.0015
6,0.25,1.0,0.7058,0.6418,0.0224,0.0015
11,0.50,2.0,0.7056,0.6418,0.0224,0.0013
9,0.50,0.5,0.7058,0.6418,0.0224,0.0015
7,0.25,2.0,0.7056,0.6418,0.0224,0.0013
8,0.50,0.0,0.7050,0.6418,0.0224,0.0008
5,0.25,0.5,0.7058,0.6418,0.0224,0.0015
4,0.25,0.0,0.7050,0.6418,0.0224,0.0008


## 11. 확신 등급 기준 (dev만)

등급 임계값은 원본 dev의 교차검증 확률로 계산한다. 여기서는 test를 읽거나 점수화하지 않는다.

In [17]:
# 2차/3차 공통 기반 모델의 교차검증 확률로 등급 경계를 고정한다.
if FINAL_2.startswith('결합 B'):
    pd_dev_2 = np.zeros((dev.sum(), 3))
    for tr, va in cv.split(Xd, yd, gd):
        pd_dev_2[va] = LogisticRegression(C=BEST_C, max_iter=2000, class_weight='balanced').fit(Xd[tr], yd[tr]).predict_proba(Xd[va])
else:
    pd_dev_2 = combine(W, A, H, B, dev)
pd_dev_3 = adjust(pd_dev_2, GS[dev], GD[dev])
ths = np.round(np.arange(0.34, 0.99, 0.01), 2)
def min_threshold_for(p_dev, target):
    for t in ths:
        keep = p_dev.max(1) >= t
        if keep.sum() >= 30 and accuracy_score(yd[keep], p_dev.argmax(1)[keep]) >= target:
            return float(t)
    return float(ths[-1])
TH_HIGH_2, TH_MID_2 = min_threshold_for(pd_dev_2, .90), min_threshold_for(pd_dev_2, .80)
TH_HIGH, TH_MID = min_threshold_for(pd_dev_3, .90), min_threshold_for(pd_dev_3, .80)
def grade(c):
    return '높음' if c >= TH_HIGH else ('보통' if c >= TH_MID else '보류')
print(f'2차 dev 임계값: 높음 {TH_HIGH_2:.2f}, 보통 {TH_MID_2:.2f} | 3차 dev 임계값: 높음 {TH_HIGH:.2f}, 보통 {TH_MID:.2f}')

2차 dev 임계값: 높음 0.85, 보통 0.67 | 3차 dev 임계값: 높음 0.85, 보통 0.67


## 13. 실사용 함수 `analyze(문장, 대상)`
에이전트가 부를 함수다. 판정과 함께 **근거**(쓴 구절, 규칙 적중어, 모델별 확률)를 돌려준다.
확신이 임계값보다 낮으면 `자료 없음`이다.

아래 예문은 **시연용으로 직접 만든 문장**이다. 실제 기사·공시가 아니며 평가에 쓰지 않는다.

In [18]:
def analyze(text, aspect, gov_adjust=True):
    text, aspect = norm(text), norm(aspect)
    cl, cut = target_clause(text, aspect)
    row = pd.DataFrame([{'text': text, 'aspect': aspect, 'clause': cl, 'clause_cut': cut,
                         'type': 'multiple' if len(clauses(text)) > 1 else 'single', **lex_features(cl)}])
    def one(bundle, a, b=None):
        tok, mdl, order = bundle
        enc = tok([a], [b] if b else None, truncation=True, max_length=192, return_tensors='pt')
        enc.pop('token_type_ids', None) if 'roberta' in mdl.config.model_type else None
        with torch.no_grad():
            return torch.softmax(mdl(**enc).logits, -1).numpy()[:, order]
    inputs = {'full': (text, None), 'clause': (cl, None), 'pair': (cl, aspect),
              'comma': (comma_clause(text, aspect)[0], None)}          # 1차 쉼표 구절도 보정 층 입력이라 함께 계산
    probs = {k: one(BUNDLE[k.split('_')[0]], *inputs[k.split('_')[1]]) for k in P if not k.startswith('nli')}
    nl = np.hstack([one(BUNDLE['nli'], cl, HYP[h].format(a=aspect)) for h in ['pos', 'neg']])  # 대상 질문 2개
    probs['nli_zero'] = nli_to3(nl)
    extra = np.hstack([nl, one(BUNDLE['info'], cl)[:, 1:2]])
    if FINAL.startswith('결합 B'):
        X = [np.log(probs[k] + 1e-9) for k in sorted(probs)]
        X += [extra, row[FEAT_LEX].values, np.array([[1.0 if row.type[0] == 'multiple' else 0.0]]),
              np.array([[float(cut)]])]
        p = LR.predict_proba(np.hstack(X))[0]
    else:
        pa, pb = probs[best_input['finbert']], probs[best_input['kofin']]
        z = np.log(W * pa + (1 - W) * pb + 1e-9)[0]
        z[2] += A * row.lex[0]; z[0] -= A * row.lex[0]; z[1] += H * (row.hedge[0] > 0) + B
        p = np.exp(z - z.max()); p /= p.sum()
    gf = gov_features(cl)                                      # 3차: 지배구조 어휘 점수
    if gov_adjust:
        p = adjust(p[None], np.array([gf['gov']]), np.array([gf['gov_deny']]))[0]
        g = grade(p.max())                                     # 11절에서 정한 등급 기준(3차)
    else:                                                      # 2차와 같은 판정(조정 없음, 2차 임계값)
        g = '높음' if p.max() >= TH_HIGH_2 else ('보통' if p.max() >= TH_MID_2 else '보류')
    label = KO[LABELS[p.argmax()]] if g != '보류' else '자료 없음'
    return {'대상': aspect, '판정': label, '원판정': KO[LABELS[p.argmax()]], '등급': g,
            '확신': round(float(p.max()), 3), '지배구조 점수': gf['gov'],
            '확률(부정/중립/긍정)': [round(float(x), 3) for x in p], '_확률_정밀': np.asarray(p, dtype=float).tolist(), '쓴 구절': cl,
            '긍정어': has(cl, POS_W), '부정어': has(cl, NEG_W), '헤지': has(cl, HEDGE)}

DEMO = [  # (시연용 예문, 대상)
    ('A전자 최대주주, 보유 지분 일부 블록딜로 매각', 'A전자'),
    ('B화학, 적자 폭 축소…3분기 흑자전환 기대', 'B화학'),
    ('C건설 대표 교체설에 회사 측 "검토한 바 없다"', 'C건설'),
    ('D제약 영업이익 12% 감소했지만 E바이오는 사상 최대 실적', 'D제약'),
    ('D제약 영업이익 12% 감소했지만 E바이오는 사상 최대 실적', 'E바이오'),
    ('F금융 이사회, 자사주 전량 소각 결의', 'F금융'),
    ('G물산 오너 일가 지분 증여…지배구조 변화는 제한적 전망', 'G물산'),
    ('H중공업 노조 파업 장기화 우려 속 수주는 순항', 'H중공업'),
]
pd.DataFrame([{'문장': t, **analyze(t, a)} for t, a in DEMO])[
    ['문장', '대상', '판정', '등급', '확신', '확률(부정/중립/긍정)', '쓴 구절']]

,문장,대상,판정,등급,확신,확률(부정/중립/긍정),쓴 구절
0,"A전자 최대주주, 보유 지분 일부 블록딜로 매각",A전자,중립,보통,0.796,"[0.06, 0.796, 0.144]","A전자 최대주주, 보유 지분 일부 블록딜로 매각"
1,"B화학, 적자 폭 축소…3분기 흑자전환 기대",B화학,긍정,높음,0.927,"[0.032, 0.041, 0.927]","B화학, 적자 폭 축소...3분기 흑자전환 기대"
2,"C건설 대표 교체설에 회사 측 ""검토한 바 없다""",C건설,자료 없음,보류,0.668,"[0.295, 0.668, 0.037]","C건설 대표 교체설에 회사 측 ""검토한 바 없다"""
3,D제약 영업이익 12% 감소했지만 E바이오는 사상 최대 실적,D제약,부정,보통,0.784,"[0.784, 0.164, 0.052]",D제약 영업이익 12% 감소했지만
4,D제약 영업이익 12% 감소했지만 E바이오는 사상 최대 실적,E바이오,긍정,보통,0.832,"[0.046, 0.122, 0.832]",E바이오는 사상 최대 실적
5,"F금융 이사회, 자사주 전량 소각 결의",F금융,중립,높음,0.947,"[0.008, 0.947, 0.046]","F금융 이사회, 자사주 전량 소각 결의"
6,G물산 오너 일가 지분 증여…지배구조 변화는 제한적 전망,G물산,중립,높음,0.924,"[0.029, 0.924, 0.047]",G물산 오너 일가 지분 증여...지배구조 변화는 제한적 전망
7,H중공업 노조 파업 장기화 우려 속 수주는 순항,H중공업,긍정,보통,0.778,"[0.121, 0.101, 0.778]",H중공업 노조 파업 장기화 우려 속 수주는 순항


## 12. 소액주주 관점 4차 어휘와 강도 조정 (전용 dev)

`data/gov_dev_set.csv`의 112문장만 사용한다. 규칙은 단어 단독 적중이 아니라 지배구조 대상과 동작의 결합으로 점수를 만든다. 10개 가상 기업을 그룹으로 나눈 교차검증으로 조정 강도와 중립 이동 강도를 고른다. 규칙 패턴은 문장별 예외 대신 사건 유형에 걸쳐 쓸 수 있게 작성한다. 점검 세트와 원본 test는 이 절에서 읽지 않는다.

In [19]:
GOV_V4_POS = [
    ('자사주 환원', r'(?:자사주|자기주식).{0,18}(?:소각|매입|취득)'),
    ('배당 환원', r'배당.{0,12}(?:확대|증액|상향|재개|인상)'),
    ('환원 이행', r'(?:주주환원|환원 정책).{0,18}(?:이행|실행|완료|강화|돌려주|마치)'),
    ('오버행·희석 해소', r'(?:오버행|물량 부담|희석(?: 우려)?|상속세 부담).{0,18}(?:해소|완화|덜|흡수|소화|상환|끝내)'),
    ('불리한 안건 철회', r'(?:유상증자|물적분할|중복상장|자회사 상장|주주에 불리한 합병).{0,20}(?:철회|취소|백지화|접기로|중단)'),
    ('증자 위협 부인', r'(?:유상증자|증자).{0,18}(?:가능성|계획|검토).{0,10}(?:일축|부인|없다고|없을)'),
    ('소액주주 권리', r'(?:소액주주|소수주주|주주).{0,16}(?:보호|권리|우선 배정|우선배정|승리|성공)'),
    ('이사회 독립성', r'(?:이사회|사외이사|감사위원|감사위원회|집중투표제|내부거래위원회).{0,20}(?:독립성|독립|분리 선출|분리선출|분리|확대|도입|강화|신설)'),
    ('혐의·소송 해소', r'(?:혐의|의혹|수사|소송).{0,18}(?:벗어|무죄|무혐의|불기소|승소|기각)'),
    ('책임·이해 일치', r'(?:경영진|대표|회장|오너).{0,18}(?:연봉 반납|사재 출연|자사주 의무 보유|책임경영)'),
    ('지배구조 개선', r'(?:승계|지배구조|순환출자).{0,18}(?:완료|안정|단순화|개선|준수율|끊어|해소)'),
    ('보상 주주 연동', r'(?:성과급|보수).{0,18}(?:주주수익률|주주 수익률|성과 연동|성과연동)'),
    ('공개매수 제안', r'공개매수.{0,18}(?:착수|제안|잔여 주식)'),
    ('대체 조달로 희석 회피', r'(?:유상증자|증자).{0,18}대신.{0,24}(?:매각|조달|자금 마련)'),
    ('상속 부담 종료', r'상속세.{0,18}(?:분할납부 끝|납부 완료|부담 해소)'),
]
GOV_V4_NEG = [
    ('자사주 처분', r'(?:자사주|자기주식).{0,18}(?:매각|처분|매도|팔|교환사채|신탁 계약 해지)'),
    ('대주주·내부자 매도', r'(?:최대주주|대주주|오너 일가|임원|회장).{0,22}(?:지분|주식).{0,14}(?:매도|매각|처분|담보|반대매매|대량매매|팔아)'),
    ('배당 후퇴', r'배당.{0,14}(?:축소|삭감|중단|동결|건너뛰|미지급)'),
    ('환원 미이행', r'(?:자사주 매입|주주환원|배당 약속|소각).{0,20}(?:연기|미루|중단|부결|미이행|지키지 않|안 해|못)'),
    ('보호예수 해제', r'보호예수.{0,14}(?:해제|풀려)'),
    ('오버행 우려', r'(?:블록딜|오버행|물량).{0,18}(?:가능성|우려|출회|부담|매각|매도)'),
    ('희석성 자금조달', r'(?:유상증자|제3자배정|전환사채|교환사채|신주인수권부사채|리픽싱|전환가).{0,20}(?:발행|조달|하향|조정|참여|상장|청구|증자)'),
    ('자회사 중복상장', r'(?:물적분할|중복상장|자회사 상장|배터리 부문).{0,20}(?:상장|추진|계획|분할)'),
    ('경영권 갈등·방어', r'(?:경영권|주주).{0,18}(?:방어|분쟁|다툼|갈등|참호|백기사|소송)'),
    ('사익·이해상충', r'(?:독립성 결여|측근|이해상충|일감 몰아주|사익편취|저가로|부당지원|부당 내부거래|자산 넘겨|불리한 합병)'),
    ('보수 인상', r'(?:보수|연봉|성과급).{0,18}(?:인상|두 배|급증)'),
    ('주주안건 저해', r'(?:주주제안|소액주주|주주).{0,18}(?:반발|부결|해임 요건 까다롭게|보호 장치 없어)'),
    ('대표 사퇴 공백', r'(?:대표|회장).{0,16}(?:사퇴|사임).{0,18}(?:공백|우려)'),
    ('제한 감사의견', r'감사의견.{0,10}(?:한정|거절)'),
    ('지배구조 문제', r'지배구조.{0,18}(?:복잡|낙후|지적)'),
    ('형사 유죄·수사', r'(?:대표|회장|오너|경영진).{0,16}(?:유죄|압수수색|기소|과징금|조사 착수)'),
    ('신주 상장·행사', r'(?:전환 청구|신주인수권 행사).{0,18}(?:신주 상장|주식 발행)'),
]
GOV_V4_DENY = [r'(?:부인|사실무근|사실 아냐|검토한 바 없|계획 없|루머|풍문)']
GOV_V4_POS_RX = [(n, re.compile(p)) for n,p in GOV_V4_POS]
GOV_V4_NEG_RX = [(n, re.compile(p)) for n,p in GOV_V4_NEG]
GOV_V4_DENY_RX = [re.compile(p) for p in GOV_V4_DENY]

def gov_features_v4(text):
    pos_hits = [n for n,rx in GOV_V4_POS_RX if rx.search(text)]
    neg_hits = [n for n,rx in GOV_V4_NEG_RX if rx.search(text)]
    deny = int(any(rx.search(text) for rx in GOV_V4_DENY_RX))
    # 같은 사건에서 겹치는 단어가 여러 번 잡혀도 한 방향당 최대 2점만 준다.
    score = min(2, len(pos_hits)) - min(2, len(neg_hits))
    # 확인되지 않은 소문·부인은 중립으로 조금 이동한다.
    return {'gov4': score, 'gov4_deny': deny, 'pos_hits': pos_hits, 'neg_hits': neg_hits}

def adjust_v4(prob, s, deny, g, d):
    z = np.log(np.asarray(prob, dtype=float) + 1e-9)
    s = np.asarray(s, dtype=float).reshape(-1)
    deny = np.asarray(deny, dtype=float).reshape(-1)
    z[:, 2] += g * s
    z[:, 0] -= g * s
    z[:, 1] += d * deny
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

DEV_GOV = pd.read_csv(os.path.join(DATA_DIR, 'gov_dev_set.csv'))
print('전용 dev:', len(DEV_GOV), '문장 /', DEV_GOV.aspect.nunique(), '개 가상 기업')

전용 dev: 112 문장 / 10 개 가상 기업


In [20]:
# 전용 dev 문장만 기준 모델에 통과시킨다. 결과를 캐시해 재실행 때 모델 추론을 반복하지 않는다.
DEV_PRED_PATH = os.path.join(OUT_DIR, 'gov_dev_predictions_4차.csv')
if os.path.exists(DEV_PRED_PATH):
    DEV_PRED = pd.read_csv(DEV_PRED_PATH)
    if len(DEV_PRED) != len(DEV_GOV) or DEV_PRED['id'].tolist() != DEV_GOV['id'].tolist():
        raise ValueError('dev 예측 캐시가 현재 dev 세트와 다릅니다. 백업 후 DEV_PRED_PATH를 지우고 다시 실행하세요.')
else:
    dev_rows = []
    for i, r in DEV_GOV.iterrows():
        a = analyze(r['text'], r['aspect'], gov_adjust=False)
        gf = gov_features_v4(a['쓴 구절'])
        dev_rows.append({'id':r['id'], 'text':r['text'], 'aspect':r['aspect'], 'label':r['label'],
                         'clause':a['쓴 구절'], 'p_neg':a['_확률_정밀'][0], 'p_neu':a['_확률_정밀'][1],
                         'p_pos':a['_확률_정밀'][2], **gf})
        if (i+1) % 20 == 0: print(f'{i+1}/{len(DEV_GOV)} dev 추론 완료')
    DEV_PRED = pd.DataFrame(dev_rows)
    DEV_PRED.to_csv(DEV_PRED_PATH, index=False, encoding='utf-8-sig')
print('4차 대상+동작 패턴 적중:', int(((DEV_PRED.gov4 != 0) | (DEV_PRED.gov4_deny != 0)).sum()), '/', len(DEV_PRED))

20/112 dev 추론 완료
40/112 dev 추론 완료
60/112 dev 추론 완료
80/112 dev 추론 완료
100/112 dev 추론 완료
4차 대상+동작 패턴 적중: 72 / 112


In [21]:
# 기업 그룹 5-fold dev 교차검증으로 어휘 강도 g와 중립 이동 d를 고른다.
y_dev_gov = DEV_PRED['label'].map({k:i for i,k in enumerate(LABELS)}).to_numpy()
p_dev_gov = DEV_PRED[['p_neg','p_neu','p_pos']].to_numpy(dtype=float)
s_dev_gov = DEV_PRED['gov4'].to_numpy(dtype=float)
dev_deny = DEV_PRED['gov4_deny'].to_numpy(dtype=float)
groups_dev_gov = DEV_PRED['aspect'].to_numpy()
cv_gov = GroupKFold(n_splits=5)
rows_v4=[]
for gg, dd in product([0,.25,.5,.75,1,1.5,2,3], [0,.25,.5,1]):
    folds=[]
    for _, va in cv_gov.split(p_dev_gov, y_dev_gov, groups_dev_gov):
        pp=adjust_v4(p_dev_gov[va], s_dev_gov[va], dev_deny[va], gg, dd)
        folds.append(f1_score(y_dev_gov[va], pp.argmax(1), average='macro'))
    rows_v4.append({'g':gg,'d':dd,'CV 평균 macro-F1':float(np.mean(folds)),
                    'CV 표준편차':float(np.std(folds, ddof=1)), 'fold 점수':folds})
TUNE_V4=pd.DataFrame(rows_v4)
base_cv=float(TUNE_V4[(TUNE_V4.g==0)&(TUNE_V4.d==0)]['CV 평균 macro-F1'].iloc[0])
eligible=TUNE_V4[TUNE_V4['CV 평균 macro-F1'] >= base_cv + .01]
if len(eligible):
    chosen=eligible.sort_values(['CV 평균 macro-F1','g','d'], ascending=[False,True,True]).iloc[0]
    G_STR4,D_STR4=float(chosen.g),float(chosen.d)
    ADOPTED4=True
else:
    G_STR4,D_STR4=0.0,0.0
    ADOPTED4=False
best_row=TUNE_V4.sort_values(['CV 평균 macro-F1','g','d'],ascending=[False,True,True]).iloc[0]
print(f'dev 그룹 CV 기준선 {base_cv:.4f}; 최고 후보 g={best_row.g}, d={best_row.d}, macro-F1={best_row["CV 평균 macro-F1"]:.4f}')
print(('채택' if ADOPTED4 else '미채택: 기준 대비 0.01 개선 없음'), f'→ g={G_STR4}, d={D_STR4}')
display(TUNE_V4.sort_values('CV 평균 macro-F1',ascending=False).head(10).drop(columns='fold 점수').round(4))

dev 그룹 CV 기준선 0.3062; 최고 후보 g=3.0, d=0.0, macro-F1=0.6825
채택 → g=3.0, d=0.0


,g,d,CV 평균 macro-F1,CV 표준편차
31,3.0,1.00,0.6825,0.1275
30,3.0,0.50,0.6825,0.1275
29,3.0,0.25,0.6825,0.1275
28,3.0,0.00,0.6825,0.1275
27,2.0,1.00,0.5839,0.1003
26,2.0,0.50,0.5839,0.1003
25,2.0,0.25,0.5839,0.1003
24,2.0,0.00,0.5839,0.1003
23,1.5,1.00,0.5226,0.0786
22,1.5,0.50,0.5226,0.0786


## 13. 최종 평가 — 이 절에서만 원본 test와 점검 세트를 연다

4차 규칙과 강도를 전용 dev로 확정한 뒤 2차·3차·4차를 같은 holdout 데이터에서 비교한다. 점검 세트는 `text`, `aspect`, `label`을 이 마지막 평가 셀에서 처음 읽는다. 정확도, macro-F1, 보류율, 판정한 문장 정확도를 함께 보고한다.

In [22]:
def metric_row(y_true, prob, high, mid):
    pred=prob.argmax(1)
    conf=prob.max(1)
    judged=conf>=mid
    return {'정확도':accuracy_score(y_true,pred), 'macro-F1':f1_score(y_true,pred,average='macro'),
            '판정 비율':float(judged.mean()),
            '판정 문장 정확도':accuracy_score(y_true[judged],pred[judged]) if judged.any() else np.nan,
            '보류 문장 수':int((~judged).sum())}

def compare_three(y_true, p2, p3, p4):
    return pd.DataFrame({
       '2차':metric_row(y_true,p2,TH_HIGH_2,TH_MID_2),
       '3차':metric_row(y_true,p3,TH_HIGH,TH_MID),
       '4차':metric_row(y_true,p4,TH_HIGH_2,TH_MID_2),
    }).T.round(4)

# 원본 KorFin-ASC test: 이 시점에서 처음 정답 점수를 계산한다.
p2_test = LR.predict_proba(features(test)) if FINAL_2.startswith('결합 B') else combine(W,A,H,B,test)
p3_test = adjust(p2_test, GS[test], GD[test])
test_v4 = [gov_features_v4(t) for t in df.loc[test,'clause']]
p4_test = adjust_v4(p2_test, [x['gov4'] for x in test_v4], [x['gov4_deny'] for x in test_v4], G_STR4,D_STR4)
y_test = df.loc[test,'y'].to_numpy()
korfin_cmp = compare_three(y_test,p2_test,p3_test,p4_test)
print('KorFin-ASC 기업 holdout test 비교')
display(korfin_cmp)

# 별도 지배구조 점검 세트는 여기서 처음 로드한다. 이 결과로 규칙·강도를 다시 바꾸지 않는다.
CH = pd.read_csv(os.path.join(DATA_DIR,'gov_challenge_set.csv'))
challenge_rows=[]
for _,r in CH.iterrows():
    a2=analyze(r['text'],r['aspect'],gov_adjust=False)
    p2=np.asarray(a2['_확률_정밀'],dtype=float)[None,:]
    clause=a2['쓴 구절']
    f3=gov_features(clause)
    p3=adjust(p2,np.array([f3['gov']]),np.array([f3['gov_deny']]))
    f4=gov_features_v4(clause)
    p4=adjust_v4(p2,[f4['gov4']],[f4['gov4_deny']],G_STR4,D_STR4)
    challenge_rows.append({'id':r['id'],'text':r['text'],'aspect':r['aspect'],'label':r['label'],
       '2차_pred':LABELS[int(p2.argmax(1)[0])],'3차_pred':LABELS[int(p3.argmax(1)[0])],
       '4차_pred':LABELS[int(p4.argmax(1)[0])], '3차_gov':f3['gov'],'4차_gov':f4['gov4'],
       '4차_pos_hits':'|'.join(f4['pos_hits']),'4차_neg_hits':'|'.join(f4['neg_hits']),
       'p2':p2[0].tolist(),'p3':p3[0].tolist(),'p4':p4[0].tolist()})
CH_RESULT=pd.DataFrame(challenge_rows)
y_ch=CH_RESULT['label'].map({k:i for i,k in enumerate(LABELS)}).to_numpy()
p2_ch=np.vstack(CH_RESULT.p2); p3_ch=np.vstack(CH_RESULT.p3); p4_ch=np.vstack(CH_RESULT.p4)
challenge_cmp=compare_three(y_ch,p2_ch,p3_ch,p4_ch)
print('지배구조 점검 세트 최종 비교 — 결과 확인 후 수정하지 않음')
display(challenge_cmp)
print('라벨별 정답 수')
display(pd.DataFrame({name:(p.argmax(1)==y_ch) for name,p in [('2차',p2_ch),('3차',p3_ch),('4차',p4_ch)]},index=CH_RESULT.label).groupby(level=0).sum().astype(int))
print('모델 간 판정이 다른 문장')
display(CH_RESULT.loc[(CH_RESULT['2차_pred']!=CH_RESULT['3차_pred']) | (CH_RESULT['2차_pred']!=CH_RESULT['4차_pred']),
 ['id','text','aspect','label','2차_pred','3차_pred','4차_pred','4차_gov','4차_pos_hits','4차_neg_hits']])

# 새 파일명으로 저장해 이전 2·3차 산출물을 덮어쓰지 않는다.
CH_RESULT.drop(columns=['p2','p3','p4']).to_csv(os.path.join(OUT_DIR,'gov_challenge_predictions_2-4차.csv'),index=False,encoding='utf-8-sig')
summary={'dev_rows':int(len(DEV_PRED)),'dev_companies':int(DEV_PRED.aspect.nunique()),
 'adopted_v4':ADOPTED4,'g_v4':G_STR4,'d_v4':D_STR4,'dev_cv_baseline_macro_f1':base_cv,
 'dev_cv_selected_macro_f1':float(TUNE_V4[(TUNE_V4.g==G_STR4)&(TUNE_V4.d==D_STR4)]['CV 평균 macro-F1'].iloc[0]),
 'korfin_test':korfin_cmp.to_dict(orient='index'),'governance_challenge':challenge_cmp.to_dict(orient='index'),
 'challenge_rows':int(len(CH)),'challenge_used_for_tuning':False}
with open(os.path.join(OUT_DIR,'sentiment_metrics_governance_4차.json'),'w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
print('저장: outputs/gov_challenge_predictions_2-4차.csv, outputs/sentiment_metrics_governance_4차.json')

KorFin-ASC 기업 holdout test 비교


,정확도,macro-F1,판정 비율,판정 문장 정확도,보류 문장 수
2차,0.7677,0.7551,0.6718,0.8683,1324.0
3차,0.7677,0.7551,0.6718,0.8675,1324.0
4차,0.7652,0.7522,0.6735,0.8646,1317.0


지배구조 점검 세트 최종 비교 — 결과 확인 후 수정하지 않음


,정확도,macro-F1,판정 비율,판정 문장 정확도,보류 문장 수
2차,0.5,0.5082,0.64,0.4062,18.0
3차,0.5,0.5128,0.70,0.4571,15.0
4차,0.6,0.5659,0.68,0.6765,16.0


라벨별 정답 수


,2차,3차,4차
label,,,
negative,10,10,16
neutral,9,8,4
positive,6,7,10


모델 간 판정이 다른 문장


,id,text,aspect,label,2차_pred,3차_pred,4차_pred,4차_gov,4차_pos_hits,4차_neg_hits
0,G01,"가나전자 최대주주, 보유 지분 3% 블록딜로 매각",가나전자,negative,neutral,neutral,negative,-2,,대주주·내부자 매도|오버행 우려
2,G03,"마바금융 이사회, 자사주 1천억원 소각 결의",마바금융,positive,neutral,neutral,positive,1,자사주 환원,
3,G04,"사아건설, 자사주 소각 계획 돌연 철회",사아건설,negative,neutral,neutral,positive,1,자사주 환원,
11,G12,자차제약 물적분할 뒤 자회사 상장 추진,자차제약,negative,neutral,neutral,negative,-1,,자회사 중복상장
13,G14,파하물산 경영권 분쟁 일단락,파하물산,positive,neutral,neutral,negative,-1,,경영권 갈등·방어
14,G15,"가나전자, 경영권 분쟁 재점화",가나전자,negative,neutral,neutral,negative,-1,,경영권 갈등·방어
17,G18,사아건설 전환사채 500억원 발행,사아건설,negative,neutral,neutral,negative,-1,,희석성 자금조달
18,G19,자차제약 전환사채 발행 계획 철회,자차제약,positive,neutral,neutral,negative,-1,,희석성 자금조달
20,G21,"파하물산, 일감 몰아주기로 과징금 부과",파하물산,negative,neutral,neutral,negative,-1,,사익·이해상충
22,G23,"다라화학 최대주주 지분 매각설…회사 ""검토한 바 없다""",다라화학,neutral,neutral,neutral,negative,-1,,대주주·내부자 매도


저장: outputs/gov_challenge_predictions_2-4차.csv, outputs/sentiment_metrics_governance_4차.json
